# Study 960 — The Unstaked ETF — the teardown

The tracking-difference estimator and its resolution limit, i.i.d. vs HAC vs block bootstrap on a negatively autocorrelated difference, the same-close benchmark that rescues the measurement, the wrapper-cost ledger and its assumption sweeps, the borrow sweep on the capture trade, the excess-of-cash race, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `4e34c697361e`), ETF era 2024-07-23 → 2026-06-30, 486 sessions.

In [1]:
R = {'start': '2024-07-23', 'end': '2026-06-30', 'n_days': 486, 'fp': '4e34c697361e', 'etha_td': 0.27, 'etha_geo': -0.31, 'etha_lo': -8.86, 'etha_hi': 9.34, 'etha_mt': 0.11, 'etha_hac': 0.03, 'etha_iid': 0.01, 'etha_res': 9.1, 'ethe_td': -1.37, 'ethe_geo': -1.31, 'ethe_lo': -10.53, 'ethe_hi': 7.77, 'ethe_mt': -0.56, 'ethe_hac': -0.16, 'ethe_res': 9.15, 'coin_sd_bps': 190, 'coin_ac1': -0.56, 'res_sweep': [('block 5', 18.0), ('block 10', 13.2), ('block 21', 9.1), ('block 42', 6.1), ('block 63', 5.4), ('monthly', 5.0)], 'res_min': 5.0, 'res_max': 18.0, 'cohort_n': 9, 'cohort_missing': 'FETH, ETHW, ETHV, QETH, EZET, CETH', 'ff_td': -1.64, 'ff_geo': -1.0, 'ff_lo': -2.27, 'ff_hi': -0.97, 'ff_mt': -5.16, 'ff_iid': -1.32, 'ff_hac': -2.89, 'ff_sd_bps': 11, 'ff_months_neg': 22, 'ff_months_n': 24, 'ff_jack_lo': -1.79, 'ff_jack_hi': -1.51, 'ff_block_hw': [1.15, 0.84, 0.65, 0.61, 0.59], 'ff_monthly_hw': 0.62, 'ff_terminal_pct': 3.1, 'ff_years': 1.92, 'fee_spread': 2.25, 'ff_share_of_fee': 73, 'fee_etha': 0.25, 'fee_ethe': 2.5, 'stake': 3.0, 'etha_short': -2.73, 'etha_share': 92, 'etha_unexpl': 0.52, 'ethe_short': -4.37, 'ethe_share': 55, 'ethe_unexpl': 1.13, 'sweep': [(2.0, -1.73, 89), (2.5, -2.23, 91), (3.0, -2.73, 92), (3.5, -3.23, 93), (4.5, -4.23, 95)], 'fee_sweep': [(0.12, 96), (0.25, 92)], 'era_e_ff': -2.43, 'era_e_ff_lo': -3.51, 'era_e_ff_hi': -1.39, 'era_e_ff_mt': -6.25, 'era_e_n': 235, 'era_l_ff': -0.9, 'era_l_ff_lo': -1.72, 'era_l_ff_hi': -0.02, 'era_l_ff_mt': -2.02, 'era_l_n': 250, 'era_e_etha': 0.36, 'era_l_etha': 0.19, 'era_e_ethe': -2.07, 'era_l_ethe': -0.71, 'cal': [(2024, -2.14, 112), (2025, -1.97, 250), (2026, -0.52, 123)], 'ls_gross': 1.85, 'ls_hac': 3.32, 'ls_iid': 1.5, 'ls_vol': 1.72, 'ls_mt': 5.69, 'ls_months_pos': 21, 'ls_months_n': 24, 'borrow': [(0, 1.45), (50, 0.95), (100, 0.45), (200, -0.55), (400, -2.55), (800, -6.55)], 'borrow_be': 145, 'sh_etha': -0.26, 'sh_ethe': -0.29, 'sh_coin': -0.299, 'adv_etha': 0.038, 'adv_etha_t': 0.24, 'fund_vol': 72.5, 'syn_planted': 2.25, 'syn_ff': -2.24, 'syn_ff_lo': -3.05, 'syn_ff_hi': -1.45, 'syn_ff_mt': -5.84, 'syn_fc': 0.28, 'syn_fc_lo': -12.01, 'syn_fc_hi': 12.95, 'syn_ratio': 16, 'syn_null_mean': -0.011, 'syn_null_sd': 0.065, 'syn_null_fire': 0}

## 1. Estimator and sample

Tracking difference is measured as the mean daily **log**-return difference × 252, so it telescopes to the endpoint wealth ratio. Both funds distribute nothing (`auto_adjust` close = price = total return); **ETH-USD is price-only and unstaked**; **BIL** is total return and is the cash leg.

**Survivorship and coverage, separately.** *Survivorship:* both lines have traded continuously since day one and the fee ranking was published before launch, so nothing is selected on an outcome. *Coverage, which is the real caveat:* the 2024-07-23 cohort has about **9** members and this study measures **two** — the two in the desk cache (FETH, ETHW, ETHV, QETH, EZET, CETH are absent) — and they are the cohort's **cheapest and dearest**, i.e. its widest fee gap. Study 959 runs the ten-wrapper cross-section on bitcoin; there is no such replication here.

> 💡 **In plain words:** we are asking how fast the fund falls behind the coin, per year, and how sure we are of that number.

In [2]:
print(f"ETHA vs ETH-USD : {R['etha_td']:+.2f}%/yr log (simple-ret diff {R['etha_geo']:+.2f}%)  "
      f"95% CI [{R['etha_lo']:+.2f}, {R['etha_hi']:+.2f}]")
print(f"   daily sd {R['coin_sd_bps']} bps  ac(1) {R['coin_ac1']:+.2f}  "
      f"iid t {R['etha_iid']:+.2f}  HAC t {R['etha_hac']:+.2f}  monthly t {R['etha_mt']:+.2f}")
print(f"ETHE vs ETH-USD : {R['ethe_td']:+.2f}%/yr log (simple-ret diff {R['ethe_geo']:+.2f}%)  "
      f"95% CI [{R['ethe_lo']:+.2f}, {R['ethe_hi']:+.2f}]  monthly t {R['ethe_mt']:+.2f}")
print('\nresolution limit at |t|=2 IS RULER-DEPENDENT -- the whole sweep:')
for label, hw in R['res_sweep']:
    print(f"   {label:>9s}: +/-{hw:5.1f}%/yr   ({hw/R['stake']:.1f}x the "
          f"{R['stake']:.1f}%/yr drag under discussion)")
print(f"-> quote the RANGE +/-{R['res_min']:.1f} to +/-{R['res_max']:.1f}%/yr. "
      'Every ruler exceeds the effect, which is the only reason the claim stands.')

ETHA vs ETH-USD : +0.27%/yr log (simple-ret diff -0.31%)  95% CI [-8.86, +9.34]
   daily sd 190 bps  ac(1) -0.56  iid t +0.01  HAC t +0.03  monthly t +0.11
ETHE vs ETH-USD : -1.37%/yr log (simple-ret diff -1.31%)  95% CI [-10.53, +7.77]  monthly t -0.56

resolution limit at |t|=2 IS RULER-DEPENDENT -- the whole sweep:
     block 5: +/- 18.0%/yr   (6.0x the 3.0%/yr drag under discussion)
    block 10: +/- 13.2%/yr   (4.4x the 3.0%/yr drag under discussion)
    block 21: +/-  9.1%/yr   (3.0x the 3.0%/yr drag under discussion)
    block 42: +/-  6.1%/yr   (2.0x the 3.0%/yr drag under discussion)
    block 63: +/-  5.4%/yr   (1.8x the 3.0%/yr drag under discussion)
     monthly: +/-  5.0%/yr   (1.7x the 3.0%/yr drag under discussion)
-> quote the RANGE +/-5.0 to +/-18.0%/yr. Every ruler exceeds the effect, which is the only reason the claim stands.


## 2. Why the interval is that wide — asynchronous closes

The coin's daily bar is stamped hours after the 16:00 fund close, so each daily difference carries a transient error that reverses the next session (Scholes-Williams / Dimson non-synchronicity, in its *late* rather than *stale* form). The error telescopes out of an endpoint comparison — which is why the two readings agree to well under a point — but it inflates every daily variance. The second column deserves its label: ETHA reads +0.27%/yr as a mean daily **log** difference and −0.31%/yr as a difference of annualised **simple** returns. That 0.6 pp swing is two things at once — the annualisation convention is level-dependent on an asset that halved, and the endpoint version is hostage to premium/discount noise on exactly two days. The log measure is the one whose sum is terminal wealth, so it is the one we quote.

> 💡 **In plain words:** the two prices are photographed at different moments, so most of what looks like tracking error is just the clock.

## 3. The same-close benchmark — and which *t* we are quoting

ETHE − ETHA: identical asset, identical closing bell, so the timing term is differenced away. The three variance treatments then disagree by a factor of four, and they are **not** equally defensible — nor is the one we quote the conservative one, so here is the direction stated plainly:

- **i.i.d. daily *t* = −1.32** — the widest ruler, and it does *not* clear \|*t*\| = 2.
- **HAC *t* = −2.89** — a *negative* autocovariance **shrinks** the HAC variance below i.i.d., so HAC is the **flattering** direction here, not a safety margin.
- **Non-overlapping monthly *t* = −5.16** and the block bootstrap — flattering in the same direction, and what the verdict is read from.

The case for discarding the i.i.d. reading is that the daily difference carries a **−0.47 one-session reversal** (premium/discount noise, essentially MA(1)): the variance of its *k*-day sum grows far more slowly than *k*, so the i.i.d. daily *t* assumes away a mechanical feature of the data and is over-wide rather than safe. Three checks decide whether that choice is doing the work — month signs, a month-jackknife, and a block sweep — and all three are printed below.

In [3]:
print(f"ETHE - ETHA: {R['ff_td']:+.2f}%/yr log (simple-return diff {R['ff_geo']:+.2f}%, "
      f"same fact: {R['ff_terminal_pct']}% of terminal wealth over {R['ff_years']} yrs)")
print(f"   95% CI [{R['ff_lo']:+.2f}, {R['ff_hi']:+.2f}]  "
      f"daily sd {R['ff_sd_bps']} bps (vs {R['coin_sd_bps']} against the coin)")
print(f"   iid t {R['ff_iid']:+.2f} (over-wide) | HAC t {R['ff_hac']:+.2f} (flattering) "
      f"| monthly t {R['ff_mt']:+.2f} <- quoted")
print(f"   robustness: {R['ff_months_neg']}/{R['ff_months_n']} months negative; "
      f"month-jackknife {R['ff_jack_lo']:+.2f} to {R['ff_jack_hi']:+.2f}%/yr")
print('   block sweep of the half-width (b=5,10,21,42,63): '
      + ', '.join(f'{h:.2f}' for h in R['ff_block_hw'])
      + f", monthly {R['ff_monthly_hw']:.2f} -> excludes zero on EVERY ruler")
print(f"   documented fee spread -{R['fee_spread']:.2f}%/yr -> measured is "
      f"{R['ff_share_of_fee']}% of it, and -{R['fee_spread']:.2f} lies inside the CI")

ETHE - ETHA: -1.64%/yr log (simple-return diff -1.00%, same fact: 3.1% of terminal wealth over 1.92 yrs)
   95% CI [-2.27, -0.97]  daily sd 11 bps (vs 190 against the coin)
   iid t -1.32 (over-wide) | HAC t -2.89 (flattering) | monthly t -5.16 <- quoted
   robustness: 22/24 months negative; month-jackknife -1.79 to -1.51%/yr
   block sweep of the half-width (b=5,10,21,42,63): 1.15, 0.84, 0.65, 0.61, 0.59, monthly 0.62 -> excludes zero on EVERY ruler
   documented fee spread -2.25%/yr -> measured is 73% of it, and -2.25 lies inside the CI


## 4. The wrapper-cost ledger

`wrapper_ledger` takes the MEASURED tracking difference and adds two declared ASSUMPTIONS: the sponsor fee and the net staking yield. The key identity is that ETH-USD is an **unstaked** price, so the staking term is *additive relative to a self-staking holder* and cannot appear in the measured column at all.

> 💡 **In plain words:** the tape can price the fee. The staking cost has to be quoted from outside, and we say so every time we quote it.

In [4]:
print(f"{'fund':6s} {'measured TD':>12s} {'fee(A)':>8s} {'stake(A)':>9s} "
      f"{'vs staked':>11s} {'stake share':>12s} {'unexplained':>12s}")
for tk, td, fee, sh, un in (("ETHA", R['etha_td'], R['fee_etha'], R['etha_share'], R['etha_unexpl']),
                            ("ETHE", R['ethe_td'], R['fee_ethe'], R['ethe_share'], R['ethe_unexpl'])):
    tot = td - R['stake']
    print(f"{tk:6s} {td:+11.2f}% {fee:7.2f}% {R['stake']:8.2f}% {tot:+10.2f}% {sh:11d}% {un:+11.2f}%")
print('\n(A) = ASSUMPTION, not tape. "unexplained" = measured TD minus the posted fee;')
print(f"both sit far inside even the NARROWEST resolution limit "
      f"(+/-{R['res_min']:.1f}%/yr, monthly ruler), i.e. no evidence of anything.")

fund    measured TD   fee(A)  stake(A)   vs staked  stake share  unexplained
ETHA         +0.27%    0.25%     3.00%      -2.73%          92%       +0.52%
ETHE         -1.37%    2.50%     3.00%      -4.37%          55%       +1.13%

(A) = ASSUMPTION, not tape. "unexplained" = measured TD minus the posted fee;
both sit far inside even the NARROWEST resolution limit (+/-5.0%/yr, monthly ruler), i.e. no evidence of anything.


## 5. Assumption sweeps

The staking yield is the one number in this study that comes from outside the tape, so it never gets a point value without a sweep. The fee gets one too, because ETHA's early waiver makes its effective rate a range rather than a number.

In [5]:
print('assumed staking -> ETHA shortfall vs a staking holder, and staking share of declared cost')
for y, tot, share in R['sweep']:
    print(f"   {y:.1f}%/yr  ->  {tot:+.2f}%/yr   ({share}%)")
print('\nassumed ETHA effective fee -> staking share')
for f, share in R['fee_sweep']:
    print(f"   {f:.2f}%/yr  ->  {share}%")
print('\nThe conclusion is insensitive to both, because the assumptions are the only inputs.')

assumed staking -> ETHA shortfall vs a staking holder, and staking share of declared cost
   2.0%/yr  ->  -1.73%/yr   (89%)
   2.5%/yr  ->  -2.23%/yr   (91%)
   3.0%/yr  ->  -2.73%/yr   (92%)
   3.5%/yr  ->  -3.23%/yr   (93%)
   4.5%/yr  ->  -4.23%/yr   (95%)

assumed ETHA effective fee -> staking share
   0.12%/yr  ->  96%
   0.25%/yr  ->  92%

The conclusion is insensitive to both, because the assumptions are the only inputs.


## 6. Era cut and calendar table

A fee is flat across eras; a forgone staking yield collapses when a fund starts staking. The fund-vs-coin rows stay uninformative in both halves (±11 to ±16 %/yr). The fund-vs-fund spread narrows by 1.5 pp — consistent with a fee change, a waiver expiring, or a staking policy change, and this tape cannot separate them.

In [6]:
print(f"ETHE-ETHA early (n={R['era_e_n']}): {R['era_e_ff']:+.2f}%/yr  "
      f"CI [{R['era_e_ff_lo']:+.2f}, {R['era_e_ff_hi']:+.2f}]  monthly t {R['era_e_ff_mt']:+.2f}")
print(f"ETHE-ETHA late  (n={R['era_l_n']}): {R['era_l_ff']:+.2f}%/yr  "
      f"CI [{R['era_l_ff_lo']:+.2f}, {R['era_l_ff_hi']:+.2f}]  monthly t {R['era_l_ff_mt']:+.2f}")
print(f"ETHA vs coin  early {R['era_e_etha']:+.2f}%/yr  late {R['era_l_etha']:+.2f}%/yr  (both noise)")
print(f"ETHE vs coin  early {R['era_e_ethe']:+.2f}%/yr  late {R['era_l_ethe']:+.2f}%/yr  (both noise)")
print('\ncalendar-year ETHE - ETHA:')
for y, v, n in R['cal']:
    print(f"   {y}: {v:+.2f}%/yr   ({n} sessions)")

ETHE-ETHA early (n=235): -2.43%/yr  CI [-3.51, -1.39]  monthly t -6.25
ETHE-ETHA late  (n=250): -0.90%/yr  CI [-1.72, -0.02]  monthly t -2.02
ETHA vs coin  early +0.36%/yr  late +0.19%/yr  (both noise)
ETHE vs coin  early -2.07%/yr  late -0.71%/yr  (both noise)

calendar-year ETHE - ETHA:
   2024: -2.14%/yr   (112 sessions)
   2025: -1.97%/yr   (250 sessions)
   2026: -0.52%/yr   (123 sessions)


## 7. Is the spread bankable? Long ETHA / short ETHE

Dollar-neutral, one unit a side. Weights are formed from the two published fees through day *t* and held from *t+1* — **the study's single execution lag**. Costs are 10 bps one-way × NAV per leg, in and out; the short leg pays an explicit annual borrow, swept, because a Grayscale-style trust is not general collateral. The book is dollar-neutral, so short proceeds fund the long leg and **no cash rebate is credited**. Gross reads slightly above the log tracking difference (+1.85 vs −1.64) because a daily-rebalanced arithmetic pair picks up the small variance difference between the two legs. Same ruler discipline as §3: the non-overlapping monthly *t* is quoted, HAC is flagged as the flattering direction, and the over-wide i.i.d. reading is shown too.

In [7]:
print(f"gross {R['ls_gross']:+.2f}%/yr  vol {R['ls_vol']:.2f}%")
print(f"   monthly t {R['ls_mt']:+.2f} ({R['ls_months_pos']}/{R['ls_months_n']} months "
      f"positive) <- quoted | HAC t {R['ls_hac']:+.2f} (flattering) | "
      f"iid t {R['ls_iid']:+.2f} (over-wide, under 2)")
for b, net in R['borrow']:
    flag = '  <- break-even is ~%d bps' % R['borrow_be'] if b == 200 else ''
    print(f"   borrow {b:4d} bps/yr -> net {net:+.2f}%/yr{flag}")

gross +1.85%/yr  vol 1.72%
   monthly t +5.69 (21/24 months positive) <- quoted | HAC t +3.32 (flattering) | iid t +1.50 (over-wide, under 2)
   borrow    0 bps/yr -> net +1.45%/yr
   borrow   50 bps/yr -> net +0.95%/yr
   borrow  100 bps/yr -> net +0.45%/yr
   borrow  200 bps/yr -> net -0.55%/yr  <- break-even is ~145 bps
   borrow  400 bps/yr -> net -2.55%/yr
   borrow  800 bps/yr -> net -6.55%/yr


## 8. Excess-of-cash race (both arms minus BIL)

Included and read with a straight face: both arms hold the same coin, so the race can only surface the drag, and a 72%-vol asset over two years cannot. Every Sharpe is negative — the ETF era so far has been an ETH bear market. The fund leg is charged a 5 bps one-way entry; the coin leg is charged nothing, which flatters the coin (the honest direction here).

> 💡 **In plain words:** you cannot spot a 3%-a-year fee by comparing Sharpe ratios of two things that are the same thing.

In [8]:
print(f"ETHA  excess Sharpe {R['sh_etha']:+.3f}")
print(f"ETHE  excess Sharpe {R['sh_ethe']:+.3f}")
print(f"coin  excess Sharpe {R['sh_coin']:+.3f}")
print(f"ETHA advantage vs coin {R['adv_etha']:+.4f}  HAC t {R['adv_etha_t']:+.2f}  -> noise")

ETHA  excess Sharpe -0.260
ETHE  excess Sharpe -0.290
coin  excess Sharpe -0.299
ETHA advantage vs coin +0.0380  HAC t +0.24  -> noise


## 9. Live synthetic control — the machinery is unbiased

*Synthetic data, not the real tape.* One true coin path; a vendor coin close carrying a transient timing error; two wrapper NAVs bleeding a planted drag plus premium/discount noise. The fund-vs-fund estimator must recover the planted spread with an interval clear of zero; the fund-vs-coin estimator, on the same world, must fail to. At `signal_strength=0` the wrappers are identical and the estimator must read zero.

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from unstaked import data, strategy as st
px, truth = data.synthetic_panel(signal_strength=1.0, seed=960)
d = st.synthetic_detect(px)
print(f"planted spread {truth['spread_planted_pct']:.2f}%/yr")
print(f"  fund-vs-fund {d['fund_vs_fund_pct']:+.2f}%/yr  "
      f"CI [{d['fund_vs_fund_ci'][0]:+.2f}, {d['fund_vs_fund_ci'][1]:+.2f}]  "
      f"monthly t {d['fund_vs_fund_monthly_t']:+.2f}")
print(f"  fund-vs-coin {d['fund_vs_coin_pct']:+.2f}%/yr  "
      f"CI [{d['fund_vs_coin_ci'][0]:+.2f}, {d['fund_vs_coin_ci'][1]:+.2f}]  "
      f"-> {d['fund_vs_coin_halfwidth']/d['fund_vs_fund_halfwidth']:.0f}x wider")
nulls = np.array([st.synthetic_detect(
    data.synthetic_panel(signal_strength=0.0, seed=960+s)[0])['fund_vs_fund_pct']
    for s in range(8)])
print(f"  null x8: mean {nulls.mean():+.3f}%/yr (sd {nulls.std(ddof=1):.3f}), "
      f"|est|>=0.5 in {(abs(nulls)>=0.5).sum()}/8")

planted spread 2.25%/yr
  fund-vs-fund -2.24%/yr  CI [-3.05, -1.45]  monthly t -5.84
  fund-vs-coin +0.28%/yr  CI [-12.01, +12.95]  -> 16x wider


  null x8: mean -0.011%/yr (sd 0.065), |est|>=0.5 in 0/8


## Verdict

- **Signal — Mixed.** The fund-vs-coin tracking difference asked for by the claim is **+0.27%/yr** (ETHA) with a block-bootstrap CI of **[-8.86, +9.34]** and a monthly *t* of +0.11 — no signal, and none obtainable, since ETH-USD is an **unstaked** price from which the reward is absent by construction. The same estimator against a same-close benchmark delivers a genuinely robust number: **ETHE − ETHA = -1.64%/yr**, CI **[-2.27, -0.97]**, monthly *t* = **-5.16**, recovering 73% of a documented 2.25%/yr fee gap that lies inside the interval, with 22/24 months negative and the interval clear of zero at every block length. Two things the badge is not allowed to hide: the i.i.d. daily *t* is only -1.32 (discarded for the reason in §3, not ignored), and this is **2 of ~9 cohort funds at the widest fee gap available**. Real on the fee, silent on the staking.
- **Tradability — Fragile.** Owning the cheap wrapper is worth the measured 1.64%/yr (3.1% of terminal wealth over 1.92 years) against a documented 2.25%/yr fee gap, for one trade and no borrow — but the evidence is 23 months of one asset and one fee-extreme pair, and the spread has narrowed from -2.43 to -0.90%/yr. The levered expression dies above ~145 bps of borrow. The ~3%/yr staking yield is not harvestable from inside any wrapper here.
- **Method, which is the durable output.** Report the resolution limit before the point estimate — and report it on more than one ruler. Against a 24/7 coin it runs ±5.0 to ±18.0%/yr depending on the block length, every member of that range larger than the 3%/yr effect; against a same-close sibling fund it is ±0.65%/yr and ruler-stable. The benchmark, not the estimator, decides whether the question can be answered.